# 🧪 Set Mode Test: Position vs. Velocity

This notebook exists to build intuition for the two control modes exposed by `/wmx/axis/set_mode`:

- **Position mode (`0`)** — command an absolute target, and the axis moves there and stops.
- **Velocity mode (`1`)** — command a speed, and the axis keeps spinning until told to stop.

You'll run each mode once and inspect `actual_pos` before/after so the difference shows up in real numbers, not just in theory.

*Prerequisite: `01_wmx_system_startup.ipynb` should already be running (WMX engine + EtherCAT communication active).*

#### 🔌 Python ROS 2 Client Connection

As in every notebook, this creates an `rclpy` client (`WmxClient`) so this Jupyter kernel can talk to the ROS 2 backend nodes. See Notebook 01's write-up on this pattern if you want the full explanation — the short version: `wmx.call(service_type, service_name, request)` is how every service call in this notebook works, and `wmx.axis_list` is which axes this notebook controls.

Use a unique `node_name` here so it doesn't collide with any other notebook's client running at the same time.

In [ ]:
import time
import rclpy
from wmx_r2_message.srv import SetAxis
from wmx_r2_message.msg import AxisPose, AxisVelocity, AxisState
from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_client_setmode_test')
print(f"✅ WMX Client Ready. Target Axes: {wmx.axis_list}")

# --- Helper: read the current actual position from /wmx/axis/state ---
_latest_state = {'msg': None}

def _state_callback(msg):
    _latest_state['msg'] = msg

_state_sub = wmx.create_subscription(AxisState, '/wmx/axis/state', _state_callback, 10)

def get_current_position():
    """Return the current actual_pos (list, one value per axis)."""
    _latest_state['msg'] = None
    for _ in range(20):
        rclpy.spin_once(wmx, timeout_sec=0.1)
        if _latest_state['msg'] is not None:
            return list(_latest_state['msg'].actual_pos)
    raise RuntimeError("No state received. Is /wmx/axis/state publishing?")

## Position Mode

Position mode commands are **absolute**: `target` is a coordinate to go to, not a distance to travel.

That coordinate system's zero point is defined by homing — whatever position the axis physically happens to be in when `/wmx/axis/homing` runs becomes `0`. From there, `actual_pos` is the total rotation (in radians, given this project's gear ratio setup) measured from that zero point. There's no wraparound — since soft limits are disabled in this config, the number just keeps growing (or shrinking) the more you rotate the same direction.

So sending `target = 6.28` means "go to the point 1 full turn away from wherever homing was done," regardless of where the axis currently is.

In [ ]:
axes = wmx.axis_list
zeros = [0] * len(axes)

# 1. Clear alarm
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
print(f"1) Clear Alarm: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 2. Servo On
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"2) Servo On: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 3. Homing
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/homing', req)
print(f"3) Homing: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 4. Set Position mode (0 = Position)
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"4) Set Position Mode: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

print("\n🚀 Ready. Run the next cell to move once in Position mode.")

In [ ]:
TARGET_POSITION = 10.0    # target position (radians)
MOVE_VELOCITY   = 5.0     # move speed
MOVE_ACC        = 10.0    # acceleration
MOVE_DEC        = 10.0    # deceleration
WAIT_SEC        = 3.0     # seconds to wait for the move to finish

pos_pub = wmx.create_publisher(AxisPose, '/wmx/axis/position', 1)

before = get_current_position()
print(f"Position before move: {before}")

msg = AxisPose()
msg.index = wmx.axis_list
msg.target = [TARGET_POSITION] * len(wmx.axis_list)
msg.velocity = [MOVE_VELOCITY] * len(wmx.axis_list)
msg.acc = [MOVE_ACC] * len(wmx.axis_list)
msg.dec = [MOVE_DEC] * len(wmx.axis_list)

print(f"🎯 Moving to target position {TARGET_POSITION} ...")
pos_pub.publish(msg)
time.sleep(WAIT_SEC)

after = get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"Position after move: {after}")
print(f"Distance traveled (rad): {[f'{t:.3f}' for t in traveled]}")
print(f"Distance traveled (turns): {[f'{t:.3f}' for t in turns]}")

## Velocity Mode

Velocity mode has no target coordinate at all — you command a **speed**, and the axis keeps spinning at that speed until you explicitly command `0`. There's no "arrival," so it's inherently a **relative** motion: how far the axis travels depends on how long you let it run (roughly `velocity × time`, plus the accel/decel ramps).

Unlike Position mode, there's no destination to compute ahead of time — you only find out how far it went by comparing `actual_pos` before and after.

In [ ]:
axes = wmx.axis_list
zeros = [0] * len(axes)

# 1. Clear alarm
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
print(f"1) Clear Alarm: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 2. Servo On
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"2) Servo On: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 3. Homing
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/homing', req)
print(f"3) Homing: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 4. Set Velocity mode (1 = Velocity)
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"4) Set Velocity Mode: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

print("\n🚀 Ready. Run the next cell to move once in Velocity mode.")

In [ ]:
RUN_VELOCITY = 3.0    # target speed
RUN_ACC      = 10.0   # acceleration
RUN_DEC      = 10.0   # deceleration
RUN_SEC      = 3.0    # how long to run at this speed (seconds)

vel_pub = wmx.create_publisher(AxisVelocity, '/wmx/axis/velocity', 1)

before = get_current_position()
print(f"Position before move: {before}")

msg = AxisVelocity()
msg.index = wmx.axis_list
msg.velocity = [RUN_VELOCITY] * len(wmx.axis_list)
msg.acc = [RUN_ACC] * len(wmx.axis_list)
msg.dec = [RUN_DEC] * len(wmx.axis_list)

print(f"🚀 Running at velocity {RUN_VELOCITY} for {RUN_SEC}s ...")
vel_pub.publish(msg)
time.sleep(RUN_SEC)

# Stop (velocity = 0)
msg.velocity = [0.0] * len(wmx.axis_list)
vel_pub.publish(msg)
print("🛑 Stopped.")

after = get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"Position after move: {after}")
print(f"Distance traveled (rad): {[f'{t:.3f}' for t in traveled]}")
print(f"Distance traveled (turns): {[f'{t:.3f}' for t in turns]}")
print(f"For reference, velocity x time = {RUN_VELOCITY * RUN_SEC:.2f} rad (theoretical, excludes accel/decel ramps)")

In [ ]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"Servo Off: {'✅ ' + res.message if res.success else '❌ ' + res.message}")